In [1]:
# ================================================================
# WEEK 7 — DAY 1
# RETRIEVAL-AUGMENTED GENERATION (RAG)
# Embeddings • Vector Search • FAISS • Semantic Retrieval
# ================================================================
#
# WEEK 7 PROJECT:
# RAG-Based Document Q&A System
#
# DAY 1 GOAL:
# Understand how the RETRIEVAL part of RAG works from scratch.
#
# By the end of this notebook, we will build:
#
#     Real Dataset (SQuAD)
#          ↓
#     Documents
#          ↓
#     Text Chunking
#          ↓
#     Embeddings (MiniLM)
#          ↓
#     Vector Representation
#          ↓
#     FAISS Vector Index
#          ↓
#     Semantic Search
#          ↓
#     Top-K Relevant Chunks
#          ↓
#     Basic RAG (Retrieval + LLM)
#
# CONCEPTS WE WILL LEARN:
#
# 1. What is RAG?
# 2. What is a document?
# 3. What is a text chunk?
# 4. Why do we chunk documents?
# 5. What are embeddings?
# 6. How text becomes vectors
# 7. What is semantic similarity?
# 8. Cosine similarity
# 9. What is vector search?
# 10. What is FAISS?
# 11. What is Top-K retrieval?
# 12. How retrieval fits into RAG
#
# DATASET:
# Stanford Question Answering Dataset (SQuAD)
#
# We will NOT load the entire dataset unnecessarily.
# We will use a manageable subset so this notebook remains:
#
#     Kaggle-friendly
#     Fast
#     Memory-efficient
#     GPU-friendly
#     Completely standalone
#
# MODEL:
# sentence-transformers/all-MiniLM-L6-v2
#
# A small pretrained embedding model suitable for learning
# semantic search without consuming a large amount of GPU memory.
#
# DEPENDENCY POLICY:
#
# This notebook does NOT depend on:
#     - Day 2 notebook
#     - Day 3 notebook
#     - Day 4 notebook
#     - Previous variables
#     - Previous embeddings
#     - Previous FAISS indexes
#     - Previous Kaggle sessions
#
# Run this notebook from top to bottom in a fresh Kaggle
# environment and it should work independently.
#
# ================================================================

print("=" * 70)
print("WEEK 7 — DAY 1: RAG RETRIEVAL FUNDAMENTALS")
print("=" * 70)
print()
print("Focus: Embeddings + Vector Search + FAISS")
print("Dataset: SQuAD")
print("Embedding Model: all-MiniLM-L6-v2")
print("Goal: Build semantic retrieval from scratch")
print()
print("Notebook Status: Standalone")
print("=" * 70)

WEEK 7 — DAY 1: RAG RETRIEVAL FUNDAMENTALS

Focus: Embeddings + Vector Search + FAISS
Dataset: SQuAD
Embedding Model: all-MiniLM-L6-v2
Goal: Build semantic retrieval from scratch

Notebook Status: Standalone


In [2]:
# Install required packages for Day 1
# These packages are needed for:
# - datasets: Loading SQuAD dataset
# - sentence-transformers: Generating embeddings
# - faiss-cpu: Building vector index
# - transformers: For the LLM in basic RAG

!pip install -q datasets sentence-transformers faiss-cpu transformers

print("All dependencies installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 80.4 MB/s eta 0:00:00
All dependencies installed successfully.


In [3]:
# Standard library imports
import random
import time
from typing import List, Dict, Tuple

# Data manipulation
import numpy as np
import pandas as pd

# Dataset loading
from datasets import load_dataset

# Machine learning
import torch

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector search
import faiss

# LLM for basic RAG
from transformers import pipeline

# Progress tracking
from tqdm import tqdm

print("All imports completed successfully.")
print(f"PyTorch version: {torch.__version__}")
print(f"FAISS version: {faiss.__version__}")

All imports completed successfully.
PyTorch version: 2.10.0+cu128
FAISS version: 1.15.0


In [4]:
# Set reproducibility and central configuration values

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device configuration (GPU if available)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Dataset configuration
DATASET_NAME = "squad"
N_SAMPLES = 5000

# Text chunking configuration
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

# Retrieval configuration
TOP_K = 5

# Embedding model
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("\nConfiguration loaded successfully.")
print(f"Random seed: {SEED}")
print(f"Device: {DEVICE}")
print(f"Dataset: {DATASET_NAME}")
print(f"Dataset samples: {N_SAMPLES}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Chunk overlap: {CHUNK_OVERLAP}")
print(f"Top-K: {TOP_K}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")

Using device: cuda

Configuration loaded successfully.
Random seed: 42
Device: cuda
Dataset: squad
Dataset samples: 5000
Chunk size: 300
Chunk overlap: 50
Top-K: 5
Embedding model: sentence-transformers/all-MiniLM-L6-v2


In [5]:
# Load the SQuAD training dataset
# Only the first N_SAMPLES records are loaded to keep the notebook lightweight

print("Loading SQuAD dataset...")
start_time = time.time()

dataset = load_dataset(
    DATASET_NAME,
    split=f"train[:{N_SAMPLES}]"
)

elapsed_time = time.time() - start_time

print(f"Dataset loaded successfully in {elapsed_time:.2f} seconds.")
print(f"Number of records: {len(dataset)}")
print(f"Column names: {dataset.column_names}")

Loading SQuAD dataset...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset loaded successfully in 2.72 seconds.
Number of records: 5000
Column names: ['id', 'title', 'context', 'question', 'answers']


In [6]:
# Inspect the dataset structure and display one example

print("Dataset shape:")
print(f"Rows: {len(dataset)}")
print(f"Columns: {len(dataset.column_names)}")

print("\nColumn names:")
for column in dataset.column_names:
    print(f"  - {column}")

print("\nFirst example:")
print("-" * 50)
example = dataset[0]
print(f"ID: {example['id']}")
print(f"Title: {example['title']}")
print(f"Context: {example['context'][:200]}...")
print(f"Question: {example['question']}")
print(f"Answers: {example['answers']}")

print("\nDataset statistics:")
print(f"Average context length: {np.mean([len(d['context']) for d in dataset]):.0f} characters")
print(f"Average question length: {np.mean([len(d['question']) for d in dataset]):.0f} characters")

Dataset shape:
Rows: 5000
Columns: 5

Column names:
  - id
  - title
  - context
  - question
  - answers

First example:
--------------------------------------------------
ID: 5733be284776f41900661182
Title: University_of_Notre_Dame
Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta...
Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answers: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}

Dataset statistics:
Average context length: 755 characters
Average question length: 59 characters


In [7]:
# Extract and clean documents from the dataset
# Each record contains a context (document) and a question

def clean_text(text: str) -> str:
    """
    Clean text by removing extra whitespace and normalizing.
    """
    # Remove extra whitespace
    text = " ".join(text.split())
    return text

print("Extracting documents from SQuAD dataset...")

documents = []
questions = []
answers = []

for idx, record in enumerate(dataset):
    context = record["context"]
    question = record["question"]
    answer = record["answers"]["text"][0] if record["answers"]["text"] else ""
    
    # Clean the context
    cleaned_context = clean_text(context)
    
    documents.append(cleaned_context)
    questions.append(question)
    answers.append(answer)

print(f"Extracted {len(documents)} documents")
print(f"Sample document (first 200 chars):")
print(documents[0][:200] + "...")

Extracting documents from SQuAD dataset...
Extracted 5000 documents
Sample document (first 200 chars):
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta...


In [8]:
# Split documents into smaller overlapping chunks
# Chunking is necessary because:
# 1. Embedding models have token limits
# 2. Retrieval works better on focused text segments
# 3. It helps capture specific information

def create_chunks(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """
    Split text into overlapping chunks.
    
    Args:
        text: Input text to chunk
        chunk_size: Number of characters per chunk
        overlap: Number of characters to overlap between chunks
    
    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = min(start + chunk_size, text_length)
        chunk = text[start:end]
        chunks.append(chunk)
        
        # Move start position for next chunk
        start += (chunk_size - overlap)
        
        # Break if we've reached the end
        if end == text_length:
            break
    
    return chunks

print("Creating text chunks from documents...")

all_chunks = []
chunk_to_doc_mapping = []  # Track which document each chunk belongs to

for idx, doc in enumerate(documents):
    chunks = create_chunks(doc, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(chunks)
    chunk_to_doc_mapping.extend([idx] * len(chunks))

print(f"Created {len(all_chunks)} chunks from {len(documents)} documents")
print(f"Average chunks per document: {len(all_chunks) / len(documents):.1f}")
print(f"Sample chunk (first 200 chars):")
print(all_chunks[0][:200] + "...")

Creating text chunks from documents...
Created 16624 chunks from 5000 documents
Average chunks per document: 3.3
Sample chunk (first 200 chars):
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta...


In [9]:
# Load the MiniLM embedding model
# This model converts text to 384-dimensional vectors

print("Loading embedding model...")
start_time = time.time()

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE
)

elapsed_time = time.time() - start_time

print(f"Model loaded successfully in {elapsed_time:.2f} seconds.")
print(f"Model name: {EMBEDDING_MODEL_NAME}")
print(f"Model device: {embedding_model.device}")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully in 3.39 seconds.
Model name: sentence-transformers/all-MiniLM-L6-v2
Model device: cuda:0
Embedding dimension: 384


/tmp/ipykernel_23/1679990869.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


In [10]:
# Generate embeddings for all text chunks
# This converts text to vectors that capture semantic meaning

print("Generating embeddings for all chunks...")
start_time = time.time()

# Process in batches for memory efficiency
batch_size = 32
embeddings = []

for i in tqdm(range(0, len(all_chunks), batch_size), desc="Generating embeddings"):
    batch_chunks = all_chunks[i:i+batch_size]
    batch_embeddings = embedding_model.encode(
        batch_chunks,
        batch_size=batch_size,
        device=DEVICE,
        show_progress_bar=False
    )
    embeddings.append(batch_embeddings)

# Combine all batches
embeddings = np.vstack(embeddings)

elapsed_time = time.time() - start_time

print(f"Embeddings generated successfully in {elapsed_time:.2f} seconds.")
print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Number of chunks: {embeddings.shape[0]}")

Generating embeddings for all chunks...


Generating embeddings: 100%|██████████| 520/520 [00:13<00:00, 37.46it/s]

Embeddings generated successfully in 13.89 seconds.
Embedding shape: (16624, 384)
Embedding dimension: 384
Number of chunks: 16624


In [11]:
# Build FAISS index for efficient vector search
# FAISS (Facebook AI Similarity Search) enables fast similarity search

print("Building FAISS index...")
start_time = time.time()

# Get embedding dimension
dimension = embeddings.shape[1]

# Create FAISS index using L2 (Euclidean) distance
# Alternative: faiss.IndexFlatIP for inner product (cosine similarity)
index = faiss.IndexFlatL2(dimension)

# Add embeddings to the index
index.add(embeddings.astype('float32'))

elapsed_time = time.time() - start_time

print(f"FAISS index built successfully in {elapsed_time:.2f} seconds.")
print(f"Index type: {type(index).__name__}")
print(f"Number of vectors in index: {index.ntotal}")
print(f"Vector dimension: {index.d}")

Building FAISS index...
FAISS index built successfully in 0.03 seconds.
Index type: IndexFlatL2
Number of vectors in index: 16624
Vector dimension: 384


In [12]:
# Perform semantic search using the FAISS index
# This retrieves the most relevant chunks for a given query

def semantic_search(query: str, k: int = TOP_K) -> Tuple[List[str], List[float]]:
    """
    Perform semantic search using FAISS index.
    
    Args:
        query: Search query text
        k: Number of top results to return
    
    Returns:
        Tuple of (chunks, distances)
    """
    # Generate embedding for the query
    query_embedding = embedding_model.encode(
        [query],
        device=DEVICE,
        show_progress_bar=False
    )
    
    # Search FAISS index
    distances, indices = index.search(
        query_embedding.astype('float32'),
        k
    )
    
    # Get the corresponding chunks
    retrieved_chunks = [all_chunks[idx] for idx in indices[0]]
    
    return retrieved_chunks, distances[0]

# Test semantic search with sample queries

test_queries = [
    "Who discovered gravity?",
    "What is the capital of France?",
    "How does machine learning work?",
]

print("Testing semantic search with sample queries...")
print("-" * 70)

for query in test_queries:
    print(f"\nQuery: {query}")
    chunks, distances = semantic_search(query)
    
    print(f"Top {len(chunks)} relevant chunks:")
    for i, (chunk, distance) in enumerate(zip(chunks, distances), 1):
        print(f"  {i}. Distance: {distance:.4f}")
        print(f"     Text: {chunk[:150]}...")
    print("-" * 50)

Testing semantic search with sample queries...
----------------------------------------------------------------------

Query: Who discovered gravity?
Top 5 relevant chunks:
  1. Distance: 1.2110
     Text: ed by low-pressure water, enabling him to patent the entire solar engine system by 1912....
  2. Distance: 1.2110
     Text: ed by low-pressure water, enabling him to patent the entire solar engine system by 1912....
  3. Distance: 1.2110
     Text: ed by low-pressure water, enabling him to patent the entire solar engine system by 1912....
  4. Distance: 1.2110
     Text: ed by low-pressure water, enabling him to patent the entire solar engine system by 1912....
  5. Distance: 1.2110
     Text: ed by low-pressure water, enabling him to patent the entire solar engine system by 1912....
--------------------------------------------------

Query: What is the capital of France?
Top 5 relevant chunks:
  1. Distance: 1.2057
     Text: f the original in Lourdes, France. It is very popular am

In [13]:
# Evaluate retrieval performance using the dataset's actual questions
# This measures how well our semantic search works

print("Evaluating retrieval performance...")
print("-" * 70)

# Use the first 100 questions for evaluation
eval_size = min(100, len(questions))
eval_questions = questions[:eval_size]
eval_answers = answers[:eval_size]

# Track retrieval metrics
retrieval_results = []

for i, (question, expected_answer) in enumerate(zip(eval_questions, eval_answers)):
    # Retrieve chunks for this question
    retrieved_chunks, distances = semantic_search(question)
    
    # Check if the answer appears in any retrieved chunk
    answer_found = any(expected_answer.lower() in chunk.lower() for chunk in retrieved_chunks)
    
    retrieval_results.append({
        'question_id': i,
        'question': question,
        'expected_answer': expected_answer,
        'answer_found': answer_found,
        'retrieved_chunks': retrieved_chunks,
        'distances': distances
    })

# Calculate retrieval accuracy
accuracy = np.mean([result['answer_found'] for result in retrieval_results])
precision_at_1 = np.mean([
    expected_answer.lower() in retrieval_results[i]['retrieved_chunks'][0].lower()
    for i in range(len(retrieval_results))
])

print(f"Retrieval Performance (k={TOP_K}):")
print(f"  Accuracy (Answer in Top-{TOP_K}): {accuracy:.2%}")
print(f"  Precision@1 (Answer in Top-1): {precision_at_1:.2%}")

# Show some example results
print("\nExample retrieval results:")
print("-" * 70)
for i in range(min(3, len(retrieval_results))):
    result = retrieval_results[i]
    print(f"\nQuestion: {result['question']}")
    print(f"Expected Answer: {result['expected_answer']}")
    print(f"Answer Found: {'Yes' if result['answer_found'] else 'No'}")
    if result['answer_found']:
        found_in = next(
            idx for idx, chunk in enumerate(result['retrieved_chunks'])
            if result['expected_answer'].lower() in chunk.lower()
        )
        print(f"Answer found in chunk #{found_in + 1}")
    print("-" * 50)

Evaluating retrieval performance...
----------------------------------------------------------------------
Retrieval Performance (k=5):
  Accuracy (Answer in Top-5): 54.00%
  Precision@1 (Answer in Top-1): 1.00%

Example retrieval results:
----------------------------------------------------------------------

Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Expected Answer: Saint Bernadette Soubirous
Answer Found: Yes
Answer found in chunk #1
--------------------------------------------------

Question: What is in front of the Notre Dame Main Building?
Expected Answer: a copper statue of Christ
Answer Found: No
--------------------------------------------------

Question: The Basilica of the Sacred heart at Notre Dame is beside to which structure?
Expected Answer: the Main Building
Answer Found: No
--------------------------------------------------


In [14]:
# Implement a basic RAG (Retrieval-Augmented Generation) system
# This combines retrieval with a language model to generate answers

print("Implementing Basic RAG System...")
print("-" * 70)

# Load a small LLM for generation
print("Loading language model for generation...")

# Fix: Use a larger max_length or max_new_tokens
try:
    # Try T5 first (more reliable for QA)
    llm_model_name = "google/flan-t5-small"
    
    llm = pipeline(
        "text2text-generation",
        model=llm_model_name,
        device=0 if DEVICE == "cuda" else -1
    )
    print(f"Language model loaded: {llm_model_name}")
    is_t5 = True
    
except Exception as e:
    print(f"Error loading {llm_model_name}: {e}")
    print("Trying alternative model...")
    
    # Fallback to distilgpt2
    llm_model_name = "distilgpt2"
    llm = pipeline(
        "text-generation",
        model=llm_model_name,
        device=0 if DEVICE == "cuda" else -1
    )
    print(f"Language model loaded: {llm_model_name}")
    is_t5 = False

def rag_generate(query: str, k: int = TOP_K) -> Dict:
    """
    Generate an answer using RAG.
    
    Args:
        query: User question
        k: Number of chunks to retrieve
    
    Returns:
        Dictionary with query, context, and generated answer
    """
    # 1. Retrieve relevant chunks
    retrieved_chunks, distances = semantic_search(query, k)
    
    # 2. Combine chunks into a single context (truncate to avoid token limits)
    context = " ".join(retrieved_chunks)
    
    # 3. Truncate context to a reasonable length
    max_context_length = 500  # Characters, not tokens
    if len(context) > max_context_length:
        context = context[:max_context_length] + "..."
    
    # 4. Create a prompt for the LLM
    if is_t5:
        # For T5 models (text2text-generation)
        prompt = f"Answer the question based on the context. Context: {context} Question: {query}"
        # Use max_new_tokens instead of max_length
        generated = llm(
            prompt, 
            max_new_tokens=50,  # Generate only 50 new tokens
            do_sample=False
        )[0]['generated_text']
    else:
        # For GPT-style models
        prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"
        # Truncate prompt if too long
        if len(prompt) > 400:
            prompt = prompt[:400] + "...\nAnswer:"
        
        generated = llm(
            prompt, 
            max_new_tokens=50,  # Generate only 50 new tokens
            do_sample=False,
            pad_token_id=llm.tokenizer.eos_token_id
        )[0]['generated_text']
        
        # Clean up the response
        generated = generated.replace(prompt, "").strip()
    
    return {
        'query': query,
        'context': context[:300] + "..." if len(context) > 300 else context,
        'retrieved_chunks': retrieved_chunks,
        'generated_answer': generated
    }

# Test the basic RAG system with short queries
test_queries = [
    "What is Notre Dame known for?",
    "Who discovered gravity?",
]

print("\nTesting Basic RAG System:")
print("-" * 70)

for query in test_queries:
    print(f"\nQuery: {query}")
    try:
        result = rag_generate(query)
        
        print(f"Generated Answer: {result['generated_answer']}")
        print(f"Number of chunks retrieved: {len(result['retrieved_chunks'])}")
        print(f"Context length: {len(result['context'])} characters")
    except Exception as e:
        print(f"Error generating answer: {e}")
        # Fallback to simple retrieval
        print("Using retrieval-only fallback...")
        chunks, _ = semantic_search(query)
        print(f"Top chunk: {chunks[0][:150]}...")
    print("-" * 50)

print("\nBasic RAG System Test Complete.")

Implementing Basic RAG System...
----------------------------------------------------------------------
Loading language model for generation...


config.json: 0.00B [00:00, ?B/s]

Error loading google/flan-t5-small: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"
Trying alternative model...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Language model loaded: distilgpt2

Testing Basic RAG System:
----------------------------------------------------------------------

Query: What is Notre Dame known for?


Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Answer: Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer:
Answer
Number of chunks retrieved: 5
Context length: 303 characters
--------------------------------------------------

Query: Who discovered gravity?
Generated Answer: The first solar engine was developed by the United States government in 1848. The first solar engine was developed by the United States government in 1848. The first solar engine was developed by the United States government in 1848. The first solar engine
Number of chunks retrieved: 5
Context length: 303 characters
--------------------------------------------------

Basic RAG System Test Complete.


In [15]:
# Simple RAG Implementation (No LLM Required)
# This is more reliable and works without external dependencies

print("Implementing Simple RAG System (No LLM)...")
print("-" * 70)

def simple_rag_answer(query: str, k: int = TOP_K) -> Dict:
    """
    Simple RAG without LLM - uses retrieval and simple extraction.
    
    Args:
        query: User question
        k: Number of chunks to retrieve
    
    Returns:
        Dictionary with query, context, and extracted answer
    """
    # 1. Retrieve relevant chunks
    retrieved_chunks, distances = semantic_search(query, k)
    
    # 2. Combine chunks
    context = " ".join(retrieved_chunks)
    
    # 3. Simple answer extraction
    sentences = context.split(". ")
    
    # Score sentences by relevance to query
    query_words = set(query.lower().split())
    scored_sentences = []
    
    for sentence in sentences:
        sentence_lower = sentence.lower()
        # Count how many query words appear in the sentence
        word_matches = sum(1 for word in query_words if word in sentence_lower)
        # Also check for phrase matches (exact query in sentence)
        phrase_score = 2 if query.lower() in sentence_lower else 0
        total_score = word_matches + phrase_score
        if total_score > 0:
            scored_sentences.append((sentence, total_score))
    
    # Sort by score and get best sentence
    if scored_sentences:
        scored_sentences.sort(key=lambda x: x[1], reverse=True)
        best_answer = scored_sentences[0][0]
    else:
        # If no sentence matches, use the first chunk
        best_answer = retrieved_chunks[0][:200] + "..."
    
    return {
        'query': query,
        'context': context[:300] + "..." if len(context) > 300 else context,
        'retrieved_chunks': retrieved_chunks,
        'generated_answer': best_answer,
        'method': 'simple_extraction'
    }

# Test simple RAG
print("Testing Simple RAG System:")
print("-" * 70)

test_queries = [
    "What is Notre Dame known for?",
    "Who discovered gravity?",
    "What is machine learning?",
]

for query in test_queries:
    print(f"\nQuery: {query}")
    result = simple_rag_answer(query)
    
    print(f"Extracted Answer: {result['generated_answer']}")
    print(f"Number of chunks retrieved: {len(result['retrieved_chunks'])}")
    print(f"Method: {result.get('method', 'unknown')}")
    print("-" * 50)

print("\nSimple RAG System Test Complete.")
print("This demonstrates the core RAG concept: Retrieve + Generate/Extract.")

Implementing Simple RAG System (No LLM)...
----------------------------------------------------------------------
Testing Simple RAG System:
----------------------------------------------------------------------

Query: What is Notre Dame known for?
Extracted Answer: The Notre Dame Victory March is often regarded as the most famous and recognizable collegiate fight song
Number of chunks retrieved: 5
Method: simple_extraction
--------------------------------------------------

Query: Who discovered gravity?
Extracted Answer: ed by low-pressure water, enabling him to patent the entire solar engine system by 1912....
Number of chunks retrieved: 5
Method: simple_extraction
--------------------------------------------------

Query: What is machine learning?
Extracted Answer: The artificial intelligence (AI) of enemies in Twilight Princess is more advanced than that of enemies in The Wind Waker
Number of chunks retrieved: 5
Method: simple_extraction
------------------------------------------

In [16]:
# Final robust RAG implementation with fallbacks

print("Implementing Robust RAG System...")
print("-" * 70)

def robust_rag(query: str, k: int = TOP_K) -> Dict:
    """
    Robust RAG with multiple fallback strategies.
    
    1. Try LLM-based generation
    2. Fallback to simple extraction
    3. Fallback to basic retrieval
    """
    # First, retrieve chunks
    retrieved_chunks, distances = semantic_search(query, k)
    context = " ".join(retrieved_chunks)
    
    # Strategy 1: Try LLM
    try:
        if is_t5:
            prompt = f"Answer: {query} Context: {context[:300]}"
            generated = llm(prompt, max_new_tokens=30, do_sample=False)[0]['generated_text']
        else:
            prompt = f"Context: {context[:300]}\nQuestion: {query}\nAnswer:"
            generated = llm(
                prompt, 
                max_new_tokens=30, 
                do_sample=False,
                pad_token_id=llm.tokenizer.eos_token_id
            )[0]['generated_text']
            generated = generated.replace(prompt, "").strip()
        
        return {
            'query': query,
            'context': context[:200] + "...",
            'retrieved_chunks': retrieved_chunks,
            'generated_answer': generated,
            'method': 'llm'
        }
    
    except Exception as e:
        print(f"LLM failed: {e}")
        
        # Strategy 2: Simple extraction
        try:
            sentences = context.split(". ")
            query_words = set(query.lower().split())
            scored = []
            for s in sentences:
                score = sum(1 for w in query_words if w in s.lower())
                if score > 0:
                    scored.append((s, score))
            if scored:
                scored.sort(key=lambda x: x[1], reverse=True)
                answer = scored[0][0]
            else:
                answer = retrieved_chunks[0][:150] + "..."
            
            return {
                'query': query,
                'context': context[:200] + "...",
                'retrieved_chunks': retrieved_chunks,
                'generated_answer': answer,
                'method': 'extraction'
            }
        
        except:
            # Strategy 3: Just return the top chunk
            return {
                'query': query,
                'context': context[:200] + "...",
                'retrieved_chunks': retrieved_chunks,
                'generated_answer': retrieved_chunks[0][:200] + "...",
                'method': 'retrieval_only'
            }

# Test robust RAG
print("Testing Robust RAG System:")
print("-" * 70)

test_queries = [
    "What is Notre Dame known for?",
    "Who discovered gravity?",
]

for query in test_queries:
    print(f"\nQuery: {query}")
    result = robust_rag(query)
    
    print(f"Method: {result.get('method', 'unknown')}")
    print(f"Answer: {result['generated_answer']}")
    print(f"Chunks: {len(result['retrieved_chunks'])}")
    print("-" * 50)

print("\nRobust RAG System Complete.")

Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Implementing Robust RAG System...
----------------------------------------------------------------------
Testing Robust RAG System:
----------------------------------------------------------------------

Query: What is Notre Dame known for?


Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Method: llm
Answer: Notre Dame is known for its football and football teams. The Notre Dame Victory March is often regarded as the most famous and recognizable collegiate fight song.
Chunks: 5
--------------------------------------------------

Query: Who discovered gravity?
Method: llm
Answer: The answer is that gravity is a force of gravity. Gravity is a force of gravity. Gravity is a force of gravity. Gravity is a force of
Chunks: 5
--------------------------------------------------

Robust RAG System Complete.


In [17]:
# RAG Implementation - Retrieval + Answer Extraction
# This approach combines retrieval with intelligent answer extraction
# No LLM required - more reliable and faster

print("=" * 70)
print("RAG IMPLEMENTATION: RETRIEVAL + ANSWER EXTRACTION")
print("=" * 70)

def rag_answer_extraction(query: str, k: int = TOP_K) -> Dict:
    """
    RAG system that retrieves relevant chunks and extracts the best answer.
    
    How it works:
    1. Convert query to embedding vector
    2. Search FAISS index for top-k similar chunks
    3. Combine retrieved chunks
    4. Extract the most relevant sentence(s)
    5. Return the extracted answer with source information
    
    Args:
        query: User question
        k: Number of chunks to retrieve
    
    Returns:
        Dictionary containing query, answer, source chunks, and metadata
    """
    # Step 1: Retrieve relevant chunks
    retrieved_chunks, distances = semantic_search(query, k)
    
    # Step 2: Combine chunks
    full_context = " ".join(retrieved_chunks)
    
    # Step 3: Split into sentences for extraction
    sentences = full_context.split(". ")
    
    # Step 4: Score each sentence by relevance to query
    query_words = set(query.lower().split())
    scored_sentences = []
    
    for sentence in sentences:
        sentence_lower = sentence.lower()
        # Count matching words
        word_matches = sum(1 for word in query_words if word in sentence_lower)
        # Bonus for exact phrase match
        phrase_bonus = 3 if query.lower() in sentence_lower else 0
        # Bonus for longer sentences (more informative)
        length_bonus = len(sentence) / 100  # Small bonus for length
        
        total_score = word_matches + phrase_bonus + length_bonus
        if total_score > 0:
            scored_sentences.append({
                'sentence': sentence,
                'score': total_score,
                'word_matches': word_matches,
                'phrase_match': phrase_bonus > 0
            })
    
    # Step 5: Get the best sentence
    if scored_sentences:
        scored_sentences.sort(key=lambda x: x['score'], reverse=True)
        best_sentence = scored_sentences[0]
        answer = best_sentence['sentence']
        
        # If multiple good sentences exist, combine them
        if len(scored_sentences) >= 2 and scored_sentences[1]['score'] > 1:
            answer = f"{scored_sentences[0]['sentence']}. {scored_sentences[1]['sentence']}"
    else:
        # Fallback: use the first chunk
        answer = retrieved_chunks[0][:200] + "..."
    
    return {
        'query': query,
        'answer': answer,
        'retrieved_chunks': retrieved_chunks,
        'context': full_context[:300] + "..." if len(full_context) > 300 else full_context,
        'num_chunks': len(retrieved_chunks),
        'confidence': scored_sentences[0]['score'] if scored_sentences else 0
    }

# Test the RAG system
print("\nTesting RAG Answer Extraction System:")
print("-" * 70)

test_queries = [
    "What is Notre Dame known for?",
    "Who discovered gravity?",
    "What is machine learning?",
    "Where is the Basilica of the Sacred Heart?",
]

for query in test_queries:
    print(f"\nQuery: {query}")
    result = rag_answer_extraction(query)
    
    print(f"Answer: {result['answer']}")
    print(f"Confidence Score: {result['confidence']:.2f}")
    print(f"Chunks Retrieved: {result['num_chunks']}")
    print(f"Context Preview: {result['context'][:100]}...")
    print("-" * 50)

print("\nRAG System Complete.")

RAG IMPLEMENTATION: RETRIEVAL + ANSWER EXTRACTION

Testing RAG Answer Extraction System:
----------------------------------------------------------------------

Query: What is Notre Dame known for?
Answer: The Notre Dame Victory March is often regarded as the most famous and recognizable collegiate fight song.. The Notre Dame Victory March is often regarded as the most famous and recognizable collegiate fight song
Confidence Score: 4.05
Chunks Retrieved: 5
Context Preview: d is considered one of the most famed and successful college football teams in history. Other ND tea...
--------------------------------------------------

Query: Who discovered gravity?
Answer: ed by low-pressure water, enabling him to patent the entire solar engine system by 1912.
Confidence Score: 0.88
Chunks Retrieved: 5
Context Preview: ed by low-pressure water, enabling him to patent the entire solar engine system by 1912. ed by low-p...
--------------------------------------------------

Query: What is machine

In [18]:
# Evaluate the RAG system on multiple questions
# This measures how well our retrieval + extraction pipeline performs

print("=" * 70)
print("RAG PERFORMANCE EVALUATION")
print("=" * 70)

def evaluate_rag_system(num_questions: int = 20) -> Dict:
    """
    Evaluate RAG system on a subset of SQuAD questions.
    
    Args:
        num_questions: Number of questions to evaluate
    
    Returns:
        Dictionary with evaluation metrics
    """
    print(f"\nEvaluating on {num_questions} questions...")
    
    # Use the first num_questions from the dataset
    eval_questions = questions[:num_questions]
    eval_answers = answers[:num_questions]
    
    results = []
    correct = 0
    
    for i, (question, expected_answer) in enumerate(zip(eval_questions, eval_answers)):
        # Get RAG answer
        rag_result = rag_answer_extraction(question)
        
        # Check if expected answer appears in the extracted answer
        expected_lower = expected_answer.lower()
        extracted_lower = rag_result['answer'].lower()
        
        is_correct = expected_lower in extracted_lower
        
        if is_correct:
            correct += 1
        
        results.append({
            'question': question,
            'expected': expected_answer,
            'extracted': rag_result['answer'],
            'correct': is_correct,
            'confidence': rag_result['confidence']
        })
    
    # Calculate metrics
    accuracy = correct / len(eval_questions) if eval_questions else 0
    
    return {
        'accuracy': accuracy,
        'correct': correct,
        'total': len(eval_questions),
        'results': results
    }

# Run evaluation
eval_results = evaluate_rag_system(20)

print(f"\nEvaluation Results:")
print(f"Accuracy: {eval_results['accuracy']:.2%}")
print(f"Correct: {eval_results['correct']} out of {eval_results['total']}")

print("\nSample Results:")
print("-" * 70)
for i in range(min(3, len(eval_results['results']))):
    r = eval_results['results'][i]
    print(f"\nQuestion: {r['question']}")
    print(f"Expected: {r['expected']}")
    print(f"Extracted: {r['extracted'][:150]}...")
    print(f"Correct: {r['correct']}")
    print("-" * 50)

RAG PERFORMANCE EVALUATION

Evaluating on 20 questions...

Evaluation Results:
Accuracy: 50.00%
Correct: 10 out of 20

Sample Results:
----------------------------------------------------------------------

Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Expected: Saint Bernadette Soubirous
Extracted: It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858.. It is a replica of t...
Correct: True
--------------------------------------------------

Question: What is in front of the Notre Dame Main Building?
Expected: a copper statue of Christ
Extracted: The most famous is Notre Dame Stadium, home of the Fighting Irish football team; it has been renovated several times and today it can hold more than 8...
Correct: False
--------------------------------------------------

Question: The Basilica of the Sacred heart at Notre Dame is beside to which structure?
Expected: the Main Building


In [19]:
# Complete summary of what was accomplished in Day 1

print("=" * 70)
print("DAY 1 COMPLETE: RAG RETRIEVAL FUNDAMENTALS")
print("=" * 70)

print("\nWHAT WE BUILT:")
print("  1. Loaded SQuAD dataset (5,000 samples)")
print("  2. Cleaned and prepared documents")
print("  3. Created overlapping text chunks (size=300, overlap=50)")
print("  4. Loaded MiniLM embedding model (384 dimensions)")
print("  5. Generated embeddings for all chunks")
print("  6. Built FAISS index for fast similarity search")
print("  7. Implemented semantic search with Top-K retrieval")
print("  8. Built RAG system: Retrieve + Extract")
print("  9. Evaluated retrieval and extraction performance")

print("\nKEY METRICS:")
print(f"  Total Documents: {len(documents)}")
print(f"  Total Chunks: {len(all_chunks)}")
print(f"  Embedding Dimension: {embeddings.shape[1]}")
print(f"  FAISS Index Size: {index.ntotal} vectors")
print(f"  Top-K: {TOP_K}")
print(f"  Evaluation Accuracy: {eval_results['accuracy']:.2%} (on 20 questions)")

print("\nCONCEPTS LEARNED:")
print("  1. RAG = Retrieval + Generation/Extraction")
print("  2. Documents must be chunked for efficient retrieval")
print("  3. Embeddings convert text to semantic vectors")
print("  4. FAISS enables fast nearest neighbor search")
print("  5. Semantic search finds meaning, not just keywords")
print("  6. Retrieval quality directly impacts answer quality")
print("  7. Simple extraction can be effective without LLMs")

print("\nSTANDALONE VERIFICATION:")
print("  Data Source: Hugging Face datasets (no local files)")
print("  Embeddings: Generated from scratch")
print("  FAISS Index: Built from scratch")
print("  Dependencies: Installed in notebook")
print("  Status: Can run independently in fresh Kaggle session")

print("\n" + "=" * 70)
print("DAY 1 NOTEBOOK COMPLETE")
print("Next: Day 2 - Hybrid Retrieval and Reranking")
print("=" * 70)

DAY 1 COMPLETE: RAG RETRIEVAL FUNDAMENTALS

WHAT WE BUILT:
  1. Loaded SQuAD dataset (5,000 samples)
  2. Cleaned and prepared documents
  3. Created overlapping text chunks (size=300, overlap=50)
  4. Loaded MiniLM embedding model (384 dimensions)
  5. Generated embeddings for all chunks
  6. Built FAISS index for fast similarity search
  7. Implemented semantic search with Top-K retrieval
  8. Built RAG system: Retrieve + Extract
  9. Evaluated retrieval and extraction performance

KEY METRICS:
  Total Documents: 5000
  Total Chunks: 16624
  Embedding Dimension: 384
  FAISS Index Size: 16624 vectors
  Top-K: 5
  Evaluation Accuracy: 50.00% (on 20 questions)

CONCEPTS LEARNED:
  1. RAG = Retrieval + Generation/Extraction
  2. Documents must be chunked for efficient retrieval
  3. Embeddings convert text to semantic vectors
  4. FAISS enables fast nearest neighbor search
  5. Semantic search finds meaning, not just keywords
  6. Retrieval quality directly impacts answer quality
  7. Si